# Phase Results: Multiclass Emotion Classification

**Project:** Music emotion classification  
**Target:** `Emotion_Label` (angry, calm, energetic, happy, romantic, sad)  
**Dataset:** `data/raw_dataset.csv`

we designed this notebook to produce a reproducible comparison, tuning records, uncertainty estimates, error analysis, sensitivity/robustness experiments, and files for the Results Package.

## 1. Experimental protocol

- Fixed split: 64% train, 16% validation, 20% test; stratified; seed 42.
- Primary metric: macro-F1, because each emotion class should contribute equally.
- Secondary metrics: accuracy, balanced accuracy, macro precision, macro recall, weighted F1.
- Preprocessing: median imputation + standardization for numeric features; most-frequent imputation + one-hot encoding for categorical features. All preprocessing is fitted inside each model pipeline, using training folds only.
- Tuning: randomized search on training data with stratified 5-fold CV; final model choice is checked on validation data.
- Final repeatability: repeat selected configurations over multiple seeds where the estimator supports stochastic training. Keep the data split fixed.

In [1]:
import os, sys, json, random, platform, subprocess, warnings, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RandomizedSearchCV,
    cross_validate, learning_curve
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)
from scipy.stats import randint, loguniform

warnings.filterwarnings("ignore")

SEED = 42
SEEDS_FINAL = [42, 123, 456, 789, 2026]
N_SPLITS = 5
N_ITER = 30
TARGET = "Emotion_Label"
DATA_FILE = "raw_dataset.csv"
DROP_COLS = ["Track_ID"]

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
REPORTS_DIR = ROOT / "reports" / "phase_results"
FIG_DIR = REPORTS_DIR / "figures"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Root:", ROOT)
print("Python:", sys.version.split()[0], "| sklearn:", sklearn.__version__)

Root: c:\Users\juse7\Code Creations VI\Machine-Learning\machine-learning-course-project
Python: 3.13.9 | sklearn: 1.7.2


## 2. Environment and run metadata

In [2]:
def git_commit(root):
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"],
            cwd=root, stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        return "n/a"

env_info = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
    "seed": SEED,
    "final_seeds": SEEDS_FINAL,
    "cv_folds": N_SPLITS,
    "tuning_trials_per_model": N_ITER,
    "git_commit": git_commit(ROOT),
    "split_protocol": "64/16/20; stratified; random_state=42",
    "test_set_previously_evaluated_in_prior_stage": True
}
(REPORTS_DIR / "run_metadata.json").write_text(json.dumps(env_info, indent=2))
print(json.dumps(env_info, indent=2))

{
  "timestamp": "2026-11-18T15:06:08",
  "python": "3.13.9",
  "platform": "Windows-11-10.0.26300-SP0",
  "numpy": "2.4.4",
  "pandas": "3.0.0",
  "scikit-learn": "1.7.2",
  "seed": 42,
  "final_seeds": [
    42,
    123,
    456,
    789,
    2026
  ],
  "cv_folds": 5,
  "tuning_trials_per_model": 30,
  "git_commit": "d33263e",
  "split_protocol": "64/16/20; stratified; random_state=42",
  "test_set_previously_evaluated_in_prior_stage": true
}


## 3. Dataset Pipeline

In [3]:
DATA_PATH = DATA_DIR / DATA_FILE
df = pd.read_csv(DATA_PATH)

print("Data source:", DATA_PATH)
print("Raw shape:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())

missing_drop_cols = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=missing_drop_cols)
n_duplicates = int(df.duplicated().sum())
df = df.drop_duplicates().dropna(subset=[TARGET]).reset_index(drop=True)

X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].astype(str).copy()
class_names = sorted(y.unique().tolist())
K = len(class_names)

quality = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "missing_count": X.isna().sum(),
    "missing_percent": (X.isna().mean() * 100).round(2),
    "unique_values": X.nunique()
})
display(quality)
display(y.value_counts().rename("count").to_frame().assign(
    share=(y.value_counts(normalize=True).round(4))
))
print(f"Duplicates removed: {n_duplicates}")
print(f"Rows after cleaning: {len(df)} | features: {X.shape[1]} | classes: {K}")

Data source: c:\Users\juse7\Code Creations VI\Machine-Learning\machine-learning-course-project\data\raw_dataset.csv
Raw shape: (5000, 10)


,Track_ID,Genre,Tempo,Energy,Danceability,Loudness,Acousticness,Instrumentalness,Valence,Emotion_Label
0,T00001,classical,112.44,0.394,0.374,-30.02,0.730,0.092,0.638,happy
1,T00002,blues,193.10,0.473,0.333,-15.20,0.185,0.061,0.459,sad
2,T00003,hiphop,162.48,0.855,0.176,-26.24,0.347,0.604,0.964,calm
3,T00004,disco,143.81,0.340,0.607,-55.00,0.663,0.966,0.219,romantic
4,T00005,disco,81.84,0.870,0.477,-48.87,0.482,0.503,0.588,angry


Columns: ['Track_ID', 'Genre', 'Tempo', 'Energy', 'Danceability', 'Loudness', 'Acousticness', 'Instrumentalness', 'Valence', 'Emotion_Label']


,dtype,missing_count,missing_percent,unique_values
Genre,str,0,0.0,10
Tempo,float64,0,0.0,4251
Energy,float64,0,0.0,996
Danceability,float64,0,0.0,997
Loudness,float64,0,0.0,3355
Acousticness,float64,0,0.0,991
Instrumentalness,float64,0,0.0,993
Valence,float64,0,0.0,997


,count,share
Emotion_Label,,
romantic,866,0.1732
calm,864,0.1728
angry,832,0.1664
happy,822,0.1644
energetic,816,0.1632
sad,800,0.1600


Duplicates removed: 0
Rows after cleaning: 5000 | features: 8 | classes: 6


## 4. Shared preprocessing and evaluation

## 5. Baselines and candidate models

The two naive classifiers and the linear baselines from the previous stage are retained as reference points. These candidate models are added.

## 6. Initial comparable evaluation

This runs the same stratified CV folds on the training partition for every candidate and records validation scores. We use the validation scores to decide which models/configurations to investigate further.

## 7. Hyperparameter tuning

Using randomized search with same number of sampled configurations per model if possible.

All search is performed on `X_train` using CV. Validation is kept for a separate comparison after tuning.

## 8. Comparing configurations on validation data

Kept the original linear baselines and naive benchmarks in the comparison. Added tuned estimators.

## 9. Final test evaluation



## 10. Uncertainty: bootstrap confidence intervals

bootstrap confidence interval over test observations for each finalist's pooled predictions. This estimates test-sample uncertainty conditional on a fitted model.


## 11. Main-results figure

## 12. Error analysis



### Confusion matrices

## 13. Sensitivity analysis



## 14. Robustness analysis



## 15. Ablation study



## 16. Limitations
